# Cross-Modal Representational Alignment: Analysis Pipeline

Computes **Linear Predictivity (LP)** and **Representational Similarity Analysis (RSA)** across vision, audio, and language foundation model embeddings.

**Inputs** — layer `.npy` files produced by `embedding_extraction.ipynb`, organized as:
```
dinov2/large/frame_layers/   dinov2/large/gen_layers/
dinov2/base/frame_layers/    dinov2/base/gen_layers/
beats/iter3/audio_layers/    beats/iter3_plus/audio_layers/
qwen3-1.7b/text_layers/
```
Each folder contains `ids.npy` and `layer_0.npy … layer_N.npy`.

**Outputs** — heatmap figures and result `.csv` files saved to `results/`.

**Analyses**
| Modality pair | LP | RSA |
|---|---|---|
| Vision × Language | ✓ | ✓ |
| Audio × Language | ✓ | ✓ |
| Vision × Audio | ✓ | ✓ |


## 1. Setup

In [ ]:
import gc
import itertools
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr
from tqdm.notebook import tqdm

os.makedirs("results", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# Matplotlib defaults for publication-quality figures
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
})


## 2. Analysis Functions

### 2.1 Shared utilities


In [ ]:
# ── Shared utilities ─────────────────────────────────────────────────────────

def get_device(use_gpu=True):
    return torch.device("cuda" if use_gpu and torch.cuda.is_available() else "cpu")


def to_device(data, device):
    """Move a numpy array or torch tensor to `device`."""
    if isinstance(data, np.ndarray):
        return torch.tensor(data, device=device)
    elif isinstance(data, torch.Tensor):
        return data.to(device)
    raise TypeError("Input must be a numpy array or a torch tensor.")


def normalize(X):
    """Z-score normalize columns; constant columns are set to 0."""
    mean = np.nanmean(X, axis=0)
    std  = np.nanstd(X, axis=0)
    std_safe = np.where(std == 0, 1, std)
    Xn = (X - mean) / std_safe
    Xn[np.isnan(Xn)] = 0
    return Xn


def fit_normalization(X):
    """Return column-wise mean and std from a training split."""
    mean = np.nanmean(X, axis=0)
    std  = np.nanstd(X, axis=0)
    std[std == 0] = 1
    return mean, std


def apply_normalization(X, mean, std):
    Xn = (X - mean) / std
    Xn[np.isnan(Xn)] = 0
    return Xn


def mat_to_df(matrix, index, columns):
    """
    Wrap a numpy matrix in a DataFrame with labelled axes,
    reversed row order for heatmap display (layer 0 at bottom).
    """
    return pd.DataFrame(
        matrix,
        index=[f"{index}_{i}" for i in range(matrix.shape[0])],
        columns=[f"{columns}_{j}" for j in range(matrix.shape[1])],
    ).iloc[::-1]


### 2.2 Linear Predictivity

In [ ]:
# ── Linear Predictivity ──────────────────────────────────────────────────────

def pairwise_correlation(A, B, device=None):
    """
    Compute the full pairwise Pearson correlation matrix between features of A and B.

    Args:
        A, B: (N, features) arrays.
    Returns:
        Tensor of shape (features_A, features_B).
    """
    if device is None:
        device = get_device()
    A = to_device(A, device).to(torch.float32)
    B = to_device(B, device).to(torch.float32)
    am = A - A.mean(dim=0, keepdim=True)
    bm = B - B.mean(dim=0, keepdim=True)
    return am.T @ bm / (
        torch.sqrt((am ** 2).sum(dim=0, keepdim=True)).T *
        torch.sqrt((bm ** 2).sum(dim=0, keepdim=True))
    )


def many_pairwise_correlation(A, B, device=None, chunk=10_000):
    """
    Memory-efficient diagonal mean of the pairwise correlation matrix.

    Processes features in chunks of `chunk` to avoid OOM on large embeddings.
    Assumes a one-to-one correspondence between feature dimensions of A and B.
    """
    if device is None:
        device = get_device()
    corr, lens = [], []
    for i in range(0, A.shape[1], chunk):
        seg_A = to_device(A[:, i:i + chunk], device)
        seg_B = to_device(B[:, i:i + chunk], device)
        diag = np.diag(pairwise_correlation(seg_A, seg_B, device).cpu().numpy())
        corr.append(np.nanmean(diag))
        lens.append(seg_A.shape[1])
    gc.collect()
    return np.sum(np.array(corr) * np.array(lens)) / np.sum(lens)


def bidirectional_predictivity(X, Y, train_idx, test_idx, device=None,
                                alphas=np.logspace(-8, 8, 17)):
    """
    Fit ridge regression in both directions (X→Y and Y→X) on the training split,
    evaluate on the test split, and return the mean of both correlations.

    Normalization is fitted on the training split only to prevent data leakage.
    """
    if device is None:
        device = get_device()

    X_tr, X_te = X[train_idx], X[test_idx]
    Y_tr, Y_te = Y[train_idx], Y[test_idx]

    mx, sx = fit_normalization(X_tr)
    X_tr, X_te = apply_normalization(X_tr, mx, sx), apply_normalization(X_te, mx, sx)

    my, sy = fit_normalization(Y_tr)
    Y_tr, Y_te = apply_normalization(Y_tr, my, sy), apply_normalization(Y_te, my, sy)

    reg_xy = RidgeCV(alphas=alphas).fit(X_tr, Y_tr)
    corr_xy = many_pairwise_correlation(reg_xy.predict(X_te), Y_te, device=device)

    reg_yx = RidgeCV(alphas=alphas).fit(Y_tr, X_tr)
    corr_yx = many_pairwise_correlation(reg_yx.predict(Y_te), X_te, device=device)

    del reg_xy, reg_yx
    gc.collect()
    return (corr_xy + corr_yx) / 2


def lin_pred_matrix(layers1, layers2, n_splits=5):
    """
    Compute a bidirectional LP matrix across all layer combinations,
    averaged over `n_splits` cross-validation folds.

    Args:
        layers1: list of (N, d1) arrays — rows of the output matrix.
        layers2: list of (N, d2) arrays — columns of the output matrix.
        n_splits: number of KFold splits.
    Returns:
        numpy array of shape (len(layers1), len(layers2)).
    """
    device = get_device()
    n1, n2 = len(layers1), len(layers2)
    fold_results = np.zeros((n_splits, n1, n2), dtype=np.float32)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(np.arange(layers1[0].shape[0]))):
        print(f"  Fold {fold_idx + 1}/{n_splits}")
        for i in tqdm(range(n1), desc="Layer rows", leave=False):
            for j in range(n2):
                fold_results[fold_idx, i, j] = bidirectional_predictivity(
                    layers1[i], layers2[j], train_idx, test_idx, device=device
                )

    return np.mean(fold_results, axis=0)


### 2.3 Representational Similarity Analysis (RSA)

In [ ]:
# ── RSA ──────────────────────────────────────────────────────────────────────

def compute_rdm(embeddings, metric="correlation"):
    """
    Compute a Representational Dissimilarity Matrix (RDM) from embeddings.

    Args:
        embeddings: (N, d) array.
        metric:     pairwise distance metric passed to scipy.spatial.distance.pdist.
    Returns:
        (N, N) RDM.
    """
    if embeddings.ndim > 2:
        embeddings = embeddings.reshape(embeddings.shape[0], -1)
    return squareform(pdist(embeddings, metric=metric))


def compare_rdms(rdm1, rdm2, metric="spearman"):
    """
    Compare two RDMs by correlating their upper triangles.

    Returns:
        (score, p_value)
    """
    idx = np.triu_indices(rdm1.shape[0], k=1)
    if metric == "spearman":
        return spearmanr(rdm1[idx], rdm2[idx])
    elif metric == "pearson":
        return pearsonr(rdm1[idx], rdm2[idx])
    raise ValueError("metric must be 'spearman' or 'pearson'")


def rsa_matrix(layers1, layers2, distance_metric="correlation",
               compare_metric="spearman", normalize_fn=normalize):
    """
    Compute an RSA similarity matrix across all layer combinations.

    Args:
        layers1:         list of (N, d) arrays — rows of the output matrix.
        layers2:         list of (N, d) arrays — columns of the output matrix.
        distance_metric: metric for building each RDM.
        compare_metric:  metric for comparing RDM pairs ('spearman' or 'pearson').
        normalize_fn:    optional normalization applied before RDM computation.
    Returns:
        numpy array of shape (len(layers1), len(layers2)).
    """
    if normalize_fn is not None:
        layers1 = [normalize_fn(x) for x in layers1]
        layers2 = [normalize_fn(x) for x in layers2]

    rdms1 = [compute_rdm(x, metric=distance_metric) for x in layers1]
    rdms2 = [compute_rdm(x, metric=distance_metric) for x in layers2]

    grid = np.zeros((len(layers1), len(layers2)))
    for i, r1 in enumerate(rdms1):
        for j, r2 in enumerate(rdms2):
            grid[i, j], _ = compare_rdms(r1, r2, metric=compare_metric)
    return grid


### 2.4 Visualization helpers

In [ ]:
# ── Visualization helpers ────────────────────────────────────────────────────

def _clean_ticklabels(ax, axis="both"):
    """Strip the 'prefix_' from seaborn heatmap tick labels."""
    if axis in ("x", "both"):
        ax.set_xticklabels(
            [t.get_text().split("_")[-1] for t in ax.get_xticklabels()],
            rotation=0,
        )
    if axis in ("y", "both"):
        ax.set_yticklabels(
            [t.get_text().split("_")[-1] for t in ax.get_yticklabels()],
            rotation=0,
        )


def lp_heatmap(df, title, xlabel, ylabel, ax=None,
               vmin=None, vmax=None, cmap="viridis",
               annot=True, fmt=".2f", annot_size=9, save_path=None):
    """Plot a single LP heatmap."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(df, ax=ax, vmin=vmin, vmax=vmax, cmap=cmap,
                annot=annot, fmt=fmt, annot_kws={"size": annot_size})
    ax.set_title(title)
    ax.set_xlabel(xlabel, labelpad=10)
    ax.set_ylabel(ylabel)
    _clean_ticklabels(ax)
    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()


def rsa_heatmap(df, title, xlabel, ylabel, ax=None,
                vmin=None, vmax=None, cmap="magma",
                annot=True, fmt=".2f", annot_size=9, save_path=None):
    """Plot a single RSA heatmap."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(df, ax=ax, vmin=vmin, vmax=vmax, cmap=cmap,
                annot=annot, fmt=fmt, annot_kws={"size": annot_size},
                cbar_kws={"label": "Spearman ρ"})
    ax.set_title(title)
    ax.set_xlabel(xlabel, labelpad=10)
    ax.set_ylabel(ylabel)
    _clean_ticklabels(ax)
    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()


def panel_heatmap(dfs, titles, xlabels, ylabels, nrows, ncols,
                  figsize, cmap, colorbar_label, save_path=None):
    """
    Plot a grid of heatmaps sharing a common color scale.

    Args:
        dfs:            list of DataFrames, one per subplot.
        titles/xlabels/ylabels: lists of strings, one per subplot.
        nrows, ncols:   grid dimensions.
        figsize:        figure size tuple.
        cmap:           colormap name.
        colorbar_label: label for the shared colorbar.
        save_path:      optional file path to save the figure.
    """
    vmin = min(df.min().min() for df in dfs)
    vmax = max(df.max().max() for df in dfs)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes_flat = axes.flat if nrows * ncols > 1 else [axes]

    for ax, df, title, xl, yl in zip(axes_flat, dfs, titles, xlabels, ylabels):
        sns.heatmap(df, ax=ax, vmin=vmin, vmax=vmax, cmap=cmap,
                    cbar_kws={"label": colorbar_label})
        ax.set_title(title)
        ax.set_xlabel(xl, labelpad=8)
        ax.set_ylabel(yl)
        _clean_ticklabels(ax)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


## 3. Load Embeddings

Upload the layer `.zip` archives from `embedding_extraction.ipynb` and unzip them, or mount Google Drive and adjust the paths below.


In [ ]:
# Adjust these paths if your folders are named or located differently
LAYER_PATHS = {
    "frame_large": "dinov2/large/frame_layers",
    "frame_base":  "dinov2/base/frame_layers",
    "gen_large":   "dinov2/large/gen_layers",
    "gen_base":    "dinov2/base/gen_layers",
    "beats3":      "beats/iter3/audio_layers",
    "beats3_plus": "beats/iter3_plus/audio_layers",
    "qwen3":       "qwen3-1.7b/text_layers",
}


In [ ]:
# Verify all IDs are aligned across modalities
id_arrays = {name: np.load(os.path.join(path, "ids.npy"))
             for name, path in LAYER_PATHS.items()
             if os.path.exists(os.path.join(path, "ids.npy"))}

for (a, ids_a), (b, ids_b) in itertools.combinations(id_arrays.items(), 2):
    assert np.array_equal(ids_a, ids_b), f"ID mismatch: {a} vs {b}"

n_samples = len(next(iter(id_arrays.values())))
print(f"✓ All IDs aligned  |  {n_samples} samples  |  {len(id_arrays)} modalities")


In [ ]:
def load_layers(folder):
    """Load all layer_N.npy files from a folder into a list of arrays."""
    layer_files = sorted(
        [f for f in os.listdir(folder) if f.startswith("layer_") and f.endswith(".npy")],
        key=lambda f: int(f.split("_")[1].split(".")[0])
    )
    return [np.load(os.path.join(folder, f)) for f in layer_files]


frame_large = load_layers(LAYER_PATHS["frame_large"])
frame_base  = load_layers(LAYER_PATHS["frame_base"])
gen_large   = load_layers(LAYER_PATHS["gen_large"])
gen_base    = load_layers(LAYER_PATHS["gen_base"])
beats3      = load_layers(LAYER_PATHS["beats3"])
beats3_plus = load_layers(LAYER_PATHS["beats3_plus"])
qwen3       = load_layers(LAYER_PATHS["qwen3"])

print("Loaded layer counts:")
for name, layers in [("DINOv2 Large (frames)", frame_large), ("DINOv2 Large (gen)", gen_large),
                      ("DINOv2 Base (frames)", frame_base),  ("DINOv2 Base (gen)", gen_base),
                      ("BEATs Iter3", beats3), ("BEATs Iter3+", beats3_plus),
                      ("Qwen3-1.7B", qwen3)]:
    print(f"  {name}: {len(layers)} layers, shape {layers[0].shape}")


## 4. Vision × Language Alignment

DINOv2 (Base and Large, middle frames and generated images) vs. Qwen3-1.7B.


### 4.1 Linear Predictivity

In [ ]:
print("Computing Vision × Language LP matrices...")
vision_lang_lp = {
    "gen_base":   lin_pred_matrix(gen_base,   qwen3),
    "frame_base": lin_pred_matrix(frame_base, qwen3),
    "gen_large":  lin_pred_matrix(gen_large,  qwen3),
    "frame_large":lin_pred_matrix(frame_large,qwen3),
}


In [ ]:
# Convert to DataFrames and save
vl_lp_dfs = {k: mat_to_df(v, index="vision", columns="lang")
             for k, v in vision_lang_lp.items()}

for k, df in vl_lp_dfs.items():
    df.to_csv(f"results/vl_lp_{k}.csv")

print("Saved:", [f"vl_lp_{k}.csv" for k in vl_lp_dfs])


In [ ]:
# Individual heatmaps
for (k, df), title, ylabel in zip(
    vl_lp_dfs.items(),
    ["Generated Images LP", "Middle Frames LP", "Generated Images LP", "Middle Frames LP"],
    ["DINOv2 Base", "DINOv2 Base", "DINOv2 Large", "DINOv2 Large"],
):
    lp_heatmap(df, title=title, xlabel="Qwen3-1.7B", ylabel=ylabel)


In [ ]:
# 2×2 panel figure
panel_heatmap(
    dfs    = [vl_lp_dfs["gen_base"], vl_lp_dfs["frame_base"],
              vl_lp_dfs["gen_large"], vl_lp_dfs["frame_large"]],
    titles = ["Generated Images", "Middle Frames"] * 2,
    xlabels= ["Qwen3-1.7B"] * 4,
    ylabels= ["DINOv2 Base"] * 2 + ["DINOv2 Large"] * 2,
    nrows=2, ncols=2, figsize=(16, 8),
    cmap="viridis", colorbar_label="LP Score",
    save_path="results/vision_lang_lp.png",
)


### 4.2 RSA

In [ ]:
print("Computing Vision × Language RSA matrices...")
vision_lang_rsa = {
    "gen_base":   rsa_matrix(gen_base,   qwen3),
    "frame_base": rsa_matrix(frame_base, qwen3),
    "gen_large":  rsa_matrix(gen_large,  qwen3),
    "frame_large":rsa_matrix(frame_large,qwen3),
}


In [ ]:
vl_rsa_dfs = {k: mat_to_df(v, index="vision", columns="lang")
              for k, v in vision_lang_rsa.items()}

for k, df in vl_rsa_dfs.items():
    df.to_csv(f"results/vl_rsa_{k}.csv")

# Individual heatmaps
for (k, df), title, ylabel in zip(
    vl_rsa_dfs.items(),
    ["Generated Images RSA", "Middle Frames RSA", "Generated Images RSA", "Middle Frames RSA"],
    ["DINOv2 Base", "DINOv2 Base", "DINOv2 Large", "DINOv2 Large"],
):
    rsa_heatmap(df, title=title, xlabel="Qwen3-1.7B", ylabel=ylabel)


In [ ]:
# 2×2 panel figure
panel_heatmap(
    dfs    = [vl_rsa_dfs["gen_base"], vl_rsa_dfs["frame_base"],
              vl_rsa_dfs["gen_large"], vl_rsa_dfs["frame_large"]],
    titles = ["Generated Images", "Middle Frames"] * 2,
    xlabels= ["Qwen3-1.7B"] * 4,
    ylabels= ["DINOv2 Base"] * 2 + ["DINOv2 Large"] * 2,
    nrows=2, ncols=2, figsize=(16, 8),
    cmap="magma", colorbar_label="Spearman ρ",
    save_path="results/vision_lang_rsa.png",
)


## 5. Audio × Language Alignment

BEATs (Iter3 and Iter3+) vs. Qwen3-1.7B.


### 5.1 Linear Predictivity

In [ ]:
print("Computing Audio × Language LP matrices...")
audio_lang_lp = {
    "beats3":      lin_pred_matrix(beats3,      qwen3),
    "beats3_plus": lin_pred_matrix(beats3_plus, qwen3),
}


In [ ]:
al_lp_dfs = {k: mat_to_df(v, index="audio", columns="lang")
             for k, v in audio_lang_lp.items()}

for k, df in al_lp_dfs.items():
    df.to_csv(f"results/al_lp_{k}.csv")

lp_heatmap(al_lp_dfs["beats3"],      title="Audio × Language LP", xlabel="Qwen3-1.7B", ylabel="BEATs Iter3")
lp_heatmap(al_lp_dfs["beats3_plus"], title="Audio × Language LP", xlabel="Qwen3-1.7B", ylabel="BEATs Iter3+")


In [ ]:
# 1×2 panel figure
panel_heatmap(
    dfs    = [al_lp_dfs["beats3"], al_lp_dfs["beats3_plus"]],
    titles = ["BEATs Iter3", "BEATs Iter3+"],
    xlabels= ["Qwen3-1.7B"] * 2,
    ylabels= ["BEATs Iter3", "BEATs Iter3+"],
    nrows=1, ncols=2, figsize=(16, 5),
    cmap="viridis", colorbar_label="LP Score",
    save_path="results/audio_lang_lp.png",
)


### 5.2 RSA

In [ ]:
print("Computing Audio × Language RSA matrices...")
audio_lang_rsa = {
    "beats3":      rsa_matrix(beats3,      qwen3),
    "beats3_plus": rsa_matrix(beats3_plus, qwen3),
}


In [ ]:
al_rsa_dfs = {k: mat_to_df(v, index="audio", columns="lang")
              for k, v in audio_lang_rsa.items()}

for k, df in al_rsa_dfs.items():
    df.to_csv(f"results/al_rsa_{k}.csv")

rsa_heatmap(al_rsa_dfs["beats3"],      title="Audio × Language RSA", xlabel="Qwen3-1.7B", ylabel="BEATs Iter3")
rsa_heatmap(al_rsa_dfs["beats3_plus"], title="Audio × Language RSA", xlabel="Qwen3-1.7B", ylabel="BEATs Iter3+")


In [ ]:
panel_heatmap(
    dfs    = [al_rsa_dfs["beats3"], al_rsa_dfs["beats3_plus"]],
    titles = ["BEATs Iter3", "BEATs Iter3+"],
    xlabels= ["Qwen3-1.7B"] * 2,
    ylabels= ["BEATs Iter3", "BEATs Iter3+"],
    nrows=1, ncols=2, figsize=(16, 5),
    cmap="magma", colorbar_label="Spearman ρ",
    save_path="results/audio_lang_rsa.png",
)


## 6. Vision × Audio Alignment

DINOv2 (Base and Large, middle frames and generated images) vs. BEATs (Iter3 and Iter3+).


### 6.1 Linear Predictivity

In [ ]:
print("Computing Vision × Audio LP matrices...")
vision_audio_lp = {
    "gen_base_b3":    lin_pred_matrix(gen_base,    beats3),
    "gen_base_b3p":   lin_pred_matrix(gen_base,    beats3_plus),
    "frame_base_b3":  lin_pred_matrix(frame_base,  beats3),
    "frame_base_b3p": lin_pred_matrix(frame_base,  beats3_plus),
    "gen_large_b3":   lin_pred_matrix(gen_large,   beats3),
    "gen_large_b3p":  lin_pred_matrix(gen_large,   beats3_plus),
    "frame_large_b3": lin_pred_matrix(frame_large, beats3),
    "frame_large_b3p":lin_pred_matrix(frame_large, beats3_plus),
}


In [ ]:
va_lp_dfs = {k: mat_to_df(v, index="audio", columns="vision")
             for k, v in vision_audio_lp.items()}

for k, df in va_lp_dfs.items():
    df.to_csv(f"results/va_lp_{k}.csv")


In [ ]:
# 2×4 panel — Base row, then Large row
panel_heatmap(
    dfs = [
        va_lp_dfs["gen_base_b3"],   va_lp_dfs["frame_base_b3"],
        va_lp_dfs["gen_base_b3p"],  va_lp_dfs["frame_base_b3p"],
        va_lp_dfs["gen_large_b3"],  va_lp_dfs["frame_large_b3"],
        va_lp_dfs["gen_large_b3p"], va_lp_dfs["frame_large_b3p"],
    ],
    titles = [
        "Gen Images", "Mid Frames", "Gen Images", "Mid Frames",
        "Gen Images", "Mid Frames", "Gen Images", "Mid Frames",
    ],
    xlabels = ["DINOv2 Base"] * 4 + ["DINOv2 Large"] * 4,
    ylabels = [
        "BEATs Iter3", "BEATs Iter3", "BEATs Iter3+", "BEATs Iter3+",
        "BEATs Iter3", "BEATs Iter3", "BEATs Iter3+", "BEATs Iter3+",
    ],
    nrows=2, ncols=4, figsize=(24, 9),
    cmap="viridis", colorbar_label="LP Score",
    save_path="results/vision_audio_lp.png",
)


### 6.2 RSA

In [ ]:
print("Computing Vision × Audio RSA matrices...")
vision_audio_rsa = {
    "gen_base_b3":    rsa_matrix(beats3,      gen_base),
    "gen_base_b3p":   rsa_matrix(beats3_plus, gen_base),
    "frame_base_b3":  rsa_matrix(beats3,      frame_base),
    "frame_base_b3p": rsa_matrix(beats3_plus, frame_base),
    "gen_large_b3":   rsa_matrix(beats3,      gen_large),
    "gen_large_b3p":  rsa_matrix(beats3_plus, gen_large),
    "frame_large_b3": rsa_matrix(beats3,      frame_large),
    "frame_large_b3p":rsa_matrix(beats3_plus, frame_large),
}


In [ ]:
va_rsa_dfs = {k: mat_to_df(v, index="audio", columns="vision")
              for k, v in vision_audio_rsa.items()}

for k, df in va_rsa_dfs.items():
    df.to_csv(f"results/va_rsa_{k}.csv")


In [ ]:
panel_heatmap(
    dfs = [
        va_rsa_dfs["gen_base_b3"],   va_rsa_dfs["frame_base_b3"],
        va_rsa_dfs["gen_base_b3p"],  va_rsa_dfs["frame_base_b3p"],
        va_rsa_dfs["gen_large_b3"],  va_rsa_dfs["frame_large_b3"],
        va_rsa_dfs["gen_large_b3p"], va_rsa_dfs["frame_large_b3p"],
    ],
    titles = [
        "Gen Images", "Mid Frames", "Gen Images", "Mid Frames",
        "Gen Images", "Mid Frames", "Gen Images", "Mid Frames",
    ],
    xlabels = ["DINOv2 Base"] * 4 + ["DINOv2 Large"] * 4,
    ylabels = [
        "BEATs Iter3", "BEATs Iter3", "BEATs Iter3+", "BEATs Iter3+",
        "BEATs Iter3", "BEATs Iter3", "BEATs Iter3+", "BEATs Iter3+",
    ],
    nrows=2, ncols=4, figsize=(24, 9),
    cmap="magma", colorbar_label="Spearman ρ",
    save_path="results/vision_audio_rsa.png",
)


## 7. Archive Results

In [ ]:
shutil.make_archive("alignment_results", "zip", "results")
size_mb = os.path.getsize("alignment_results.zip") / 1e6
print(f"alignment_results.zip  ({size_mb:.1f} MB)")
print("Contents:")
for f in sorted(os.listdir("results")):
    print(f"  {f}")
